# Step 1.1: Parse the metadata.json files for DFDC Parts 0-6

In this step, we will locate the `metadata.json` files for each of the 7 parts (Parts 0 to 6) of the DFDC training dataset, load them, and aggregate their entries into a single unified pandas DataFrame.

In [1]:
import os
import json
import pandas as pd

# Path to the dataset root folder
dataset_dir = "dfdc_train_dataset"

# Aggregate metadata from parts 0-6
metadata_list = []

for part_idx in range(7):
    part_folder = f"dfdc_train_part_{part_idx}"
    metadata_path = os.path.join(dataset_dir, part_folder, "metadata.json")
    
    if os.path.exists(metadata_path):
        with open(metadata_path, "r") as f:
            part_data = json.load(f)
            
            for video_name, info in part_data.items():
                metadata_list.append({
                    "video_name": video_name,
                    "part": part_idx,
                    "label": info.get("label"),
                    "split": info.get("split"),
                    "original": info.get("original", None),
                    "path": os.path.join(dataset_dir, part_folder, video_name)
                })
    else:
        print(f"Warning: {metadata_path} not found.")

# Convert to a pandas DataFrame
df_metadata = pd.DataFrame(metadata_list)

# Print overview statistics
print(f"Successfully parsed {len(df_metadata)} videos across parts 0-6.")
print("\nLabel distribution:")
print(df_metadata["label"].value_counts())

# Display the first few entries
df_metadata.head()

Successfully parsed 13884 videos across parts 0-6.

Label distribution:
label
FAKE    12275
REAL     1609
Name: count, dtype: int64


,video_name,part,label,split,original,path
0,owxbbpjpch.mp4,0,FAKE,train,wynotylpnm.mp4,dfdc_train_dataset/dfdc_train_part_0/owxbbpjpc...
1,vpmyeepbep.mp4,0,REAL,train,None,dfdc_train_dataset/dfdc_train_part_0/vpmyeepbe...
2,fzvpbrzssi.mp4,0,REAL,train,None,dfdc_train_dataset/dfdc_train_part_0/fzvpbrzss...
3,htorvhbcae.mp4,0,FAKE,train,wclvkepakb.mp4,dfdc_train_dataset/dfdc_train_part_0/htorvhbca...
4,fckxaqjbxk.mp4,0,FAKE,train,vpmyeepbep.mp4,dfdc_train_dataset/dfdc_train_part_0/fckxaqjbx...


# Step 1.2: Balance the Dataset

The DFDC dataset is heavily imbalanced towards FAKE videos (e.g. 12,275 FAKEs vs 1,609 REALs in our parsed subset). To ensure our model does not learn a majority-class bias, we will write a balancing script that downsamples the FAKE videos to match the exact number of REAL videos.

In [2]:
# Separate REAL and FAKE videos
df_real = df_metadata[df_metadata["label"] == "REAL"]
df_fake = df_metadata[df_metadata["label"] == "FAKE"]

# Determine the target size (minimum count of the two)
min_count = min(len(df_real), len(df_fake))

# Sample FAKE videos to match the number of REAL videos
df_fake_balanced = df_fake.sample(n=min_count, random_state=42)

# Concatenate to get the balanced dataset
df_balanced = pd.concat([df_real, df_fake_balanced]).reset_index(drop=True)

# Print balancing results
print(f"Target size per class: {min_count}")
print(f"Total balanced dataset size: {len(df_balanced)}")
print("\nBalanced Label distribution:")
print(df_balanced["label"].value_counts())

# Display a sample of the balanced dataset
df_balanced.sample(5, random_state=42)

Target size per class: 1609
Total balanced dataset size: 3218

Balanced Label distribution:
label
REAL    1609
FAKE    1609
Name: count, dtype: int64


,video_name,part,label,split,original,path
789,xiutcafpof.mp4,4,REAL,train,None,dfdc_train_dataset/dfdc_train_part_4/xiutcafpo...
2392,txsyoxlotc.mp4,3,FAKE,train,lkwtqysqwr.mp4,dfdc_train_dataset/dfdc_train_part_3/txsyoxlot...
144,obozirvzxd.mp4,1,REAL,train,None,dfdc_train_dataset/dfdc_train_part_1/obozirvzx...
1133,mgqlxlnize.mp4,5,REAL,train,None,dfdc_train_dataset/dfdc_train_part_5/mgqlxlniz...
463,kytwyrusew.mp4,3,REAL,train,None,dfdc_train_dataset/dfdc_train_part_3/kytwyruse...


# Steps 1.3 - 1.5: Video Ingestion, Facial Detection, and Bounding Box Expansion

Here we develop the complete preprocessing pipeline to extract training face-crops from our balanced dataset:
- **Step 1.3:** OpenCV video ingestion to sample 15 equidistant frames per target video.
- **Step 1.4:** MTCNN for facial detection on the sampled frames. Only process videos where a face is detected (skip otherwise).
- **Step 1.5:** Apply bounding box expansion logic (30% margin) and save face crops as `.jpg` files organized into `train/REAL`, `train/FAKE`, `val/REAL`, and `val/FAKE` directories based on the deterministic Video-ID split rule. 
~280 min

In [3]:
import os
import cv2
import json
import hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm
from pathlib import Path

# Constants for frame extraction and face detection
OUTPUT_ROOT = "processed_faces_v2"
NUM_FRAMES = 15                  # Number of frames to sample per video
MARGIN = 0.30                    # Margin to expand bounding box by 30%
TRAIN_SPLIT_RATIO = 0.8          # Split ratio for train/validation
MIN_FACE_SIZE = 60               # Minimum size of face in pixels
USE_LARGEST_FACE_ONLY = True     # If multiple faces are detected, keep only the largest

# Setup GPU device if available (CUDA or MPS)
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Note: facenet-pytorch MTCNN/resize path can trigger unsupported adaptive pooling on MPS.
# Workaround: run detection on CPU when running on MPS to avoid runtime error.
mtcnn_device = "cpu" if device == "mps" else device
print(f"Using device: {device} for PyTorch | MTCNN running on: {mtcnn_device}")

# Initialize MTCNN detector
mtcnn = MTCNN(keep_all=True, device=mtcnn_device)

# --- Helper Functions ---

def deterministic_split(video_id, train_ratio=0.8):
    """
    Deterministically split a video into train or val based on its video_id hash.
    Ensures that Actor Leakage is prevented by grouping all frames from the same video.
    """
    h = int(hashlib.md5(video_id.encode("utf-8")).hexdigest()[:8], 16)
    return "train" if (h % 1000) / 1000.0 < train_ratio else "val"

def sample_frame_indices(frame_count, num_samples):
    """
    Equidistantly sample num_samples frame indices from the video.
    """
    if frame_count <= num_samples:
        return list(range(frame_count))
    positions = np.linspace(0, frame_count - 1, num_samples, dtype=int)
    return list(np.unique(positions))

def expand_bbox(box, img_w, img_h, margin):
    """
    Expand the detected bounding box by a given margin percentage.
    """
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    x1n = max(0, int(x1 - margin * w))
    y1n = max(0, int(y1 - margin * h))
    x2n = min(img_w, int(x2 + margin * w))
    y2n = min(img_h, int(y2 + margin * h))
    return x1n, y1n, x2n, y2n

Using device: mps for PyTorch | MTCNN running on: cpu


In [4]:
def process_video_pipeline(video_path, video_filename, label, out_root, mtcnn, num_frames=15, margin=0.30):
    """
    Ingests a video, samples equidistant frames, runs MTCNN facial detection,
    applies bounding box expansion, and saves the cropped face.
    Returns:
        int: Number of face crops saved if successful, 0 if skipped/no faces.
    """
    try:
        # Determine the target output directory based on the split rule and label
        split = deterministic_split(video_filename, TRAIN_SPLIT_RATIO)
        out_dir = os.path.join(out_root, split, label)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return 0
        
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if frame_count == 0:
            cap.release()
            return 0
        
        # Sample 15 frames equidistantly
        indices = sample_frame_indices(frame_count, num_samples=num_frames)
        saved_count = 0
        
        # Extract and process each target frame
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if not ret or frame is None:
                continue
            
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)
            
            # Detect faces
            boxes, probs = mtcnn.detect(pil_img)
            if boxes is None:
                continue
                
            valid_boxes = []
            for box in boxes:
                if box is None:
                    continue
                x1, y1, x2, y2 = box
                width, height = int(x2 - x1), int(y2 - y1)
                # Filter out faces that are too small
                if width < MIN_FACE_SIZE or height < MIN_FACE_SIZE:
                    continue
                valid_boxes.append(box)
                
            if not valid_boxes:
                continue
            
            # Select target face(s)
            if USE_LARGEST_FACE_ONLY and len(valid_boxes) > 1:
                areas = [(b[2]-b[0])*(b[3]-b[1]) for b in valid_boxes]
                idx_largest = int(np.argmax(areas))
                faces_to_use = [valid_boxes[idx_largest]]
            else:
                faces_to_use = valid_boxes
                
            # Crop and save faces
            for fi, box in enumerate(faces_to_use):
                x1, y1, x2, y2 = box
                img_w, img_h = pil_img.width, pil_img.height
                x1n, y1n, x2n, y2n = expand_bbox((x1, y1, x2, y2), img_w, img_h, margin)
                crop = pil_img.crop((x1n, y1n, x2n, y2n))
                
                # Create unique output filename
                base_name = os.path.splitext(video_filename)[0]
                out_name = f"{base_name}_f{idx:06d}"
                if len(faces_to_use) > 1:
                    out_name += f"_p{fi}"
                
                os.makedirs(out_dir, exist_ok=True)
                out_path = os.path.join(out_dir, out_name + ".jpg")
                crop.save(out_path, quality=95)
                saved_count += 1
                
        cap.release()
        return saved_count

    except Exception as e:
        # Safe logging to prevent crashes
        print(f"\n[Pipeline Exception] Failed to process {video_filename}: {e}")
        return 0

# Create the parent directories for output root
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f"Beginning ingestion pipeline for {len(df_balanced)} balanced videos...")
print(f"Saving face crops to: {OUTPUT_ROOT}\n")

processed_count = 0
skipped_count = 0
total_crops_saved = 0

# Progress bar tracking overall video progression with micro process statistics
pbar = tqdm(df_balanced.iterrows(), total=len(df_balanced), desc="Processing Videos")

for idx, row in pbar:
    video_filename = row["video_name"]
    video_path = row["path"]
    label = row["label"]
    
    # Process the video via pipeline
    crops_saved = process_video_pipeline(
        video_path=video_path,
        video_filename=video_filename,
        label=label,
        out_root=OUTPUT_ROOT,
        mtcnn=mtcnn,
        num_frames=NUM_FRAMES,
        margin=MARGIN
    )
    
    if crops_saved > 0:
        processed_count += 1
        total_crops_saved += crops_saved
    else:
        skipped_count += 1
        
    # Real-time micro statistics tracking directly on progression bar
    pbar.set_postfix({
        "Crops Saved": total_crops_saved,
        "Processed": processed_count,
        "Skipped": skipped_count
    })

print(f"\n{'='*50}")
print("Pipeline Processing Finished!")
print(f"Total Videos Processed (with faces found): {processed_count}")
print(f"Total Videos Skipped (no faces or error): {skipped_count}")
print(f"Total Face Crops Saved: {total_crops_saved}")
print(f"{'='*50}")

Beginning ingestion pipeline for 3218 balanced videos...
Saving face crops to: processed_faces_v2



Processing Videos:   0%|          | 0/3218 [00:00<?, ?it/s]


Pipeline Processing Finished!
Total Videos Processed (with faces found): 3218
Total Videos Skipped (no faces or error): 0
Total Face Crops Saved: 47579


# Verification: Count Extracted Face Crops

Once the pipeline completes, we can run this cell to count and verify the split distribution of the extracted face images in each train and validation folder.

In [5]:
import os

# Explicitly define the directory name to prevent NameError if this cell is executed independently
OUTPUT_ROOT = "processed_faces_v2"

# Check counts in output directories
for split in ["train", "val"]:
    for label in ["REAL", "FAKE"]:
        folder = os.path.join(OUTPUT_ROOT, split, label)
        count = 0
        if os.path.isdir(folder):
            count = len([f for f in os.listdir(folder) if f.lower().endswith(".jpg")])
        print(f"{split}/{label}: {count} face images")

train/REAL: 18959 face images
train/FAKE: 18953 face images
val/REAL: 4919 face images
val/FAKE: 4748 face images
